> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 05 · THE MODEL</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Sarvam-105B — reasoning, tools, streaming, caching</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Base-URL swap · every parameter · prompt caching measured · tool calling · structured output</div>
</div>

**Time:** 70 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹5 &nbsp;·&nbsp; **Prereq:** Lab 00

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## 1 · The ninety seconds that reframes the platform

Chat completions are **OpenAI-compatible**. Your existing code, LangChain chains and
Vercel AI SDK apps work with a base-URL change.

In [3]:
# Path A — the OpenAI SDK, pointed at Sarvam
from openai import OpenAI

oa = OpenAI(api_key=API_KEY, base_url="https://api.sarvam.ai/v1")

r = oa.chat.completions.create(          # note: .create() — OpenAI convention
    model="sarvam-105b",
    messages=[{"role": "user", "content": "GST का आर्थिक प्रभाव एक पैराग्राफ में।"}],
    max_tokens=2000,
)
cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
print(r.choices[0].message.content[:400])

भारत में **1 जुलाई 2017** को लागू किए गए गुड्स एंड सर्विसेज टैक्स (GST) का आर्थिक असर एक मिला-जुला लेकिन बहुत ठोस बदलाव रहा है, जिसका मकसद टैक्स सिस्टम को आसान बनाना और एक सिंगल नेशनल मार्केट बनाना था। कई पुराने टैक्सों को एक साथ मिलाकर, GST ने पुराने सिस्टम के "टैक्स-ऑन-टैक्स" वाले कैस्केडिंग असर को खत्म कर दिया, जिससे ट्रांसपेरेंसी बढ़ गई, टैक्स बेस बढ़ा, और चीजों को बनाने और बेचने का प्रोसेस आस


In [4]:
# Path B — the native Sarvam SDK
# ⚠️ NOTE THE DIFFERENCE: client.chat.completions(...)  — there is NO .create()
r2 = client.chat.completions(
    model="sarvam-105b",
    messages=[{"role": "user", "content": "GST का आर्थिक प्रभाव एक पैराग्राफ में।"}],
    max_tokens=2000,
)
cost.llm(r2.usage.prompt_tokens, r2.usage.completion_tokens)
print(r2.choices[0].message.content[:400])

भारत के लिए, गुड्स एंड सर्विसेज टैक्स (GST) एक बहुत ही अहम टैक्स सुधार था जिसका मकसद कई तरह के इनडायरेक्ट टैक्स को हटाकर एक सिंगल, यूनिफाइड नेशनल मार्केट बनाना था। लंबे समय में, यह हमारे टैक्स बेस को बड़ा करने, इकॉनमी को फॉर्मल बनाने में सफल रहा है क्योंकि एक्सचेंजों में इनपुट क्रेडिट के लिए टैक्स देने के लिए ज्यादा बिज़नेस टैक्स के दायरे में आए, और इंटरस्टेट चेक-पोस्ट्स को खत्म करके लॉजिस्टिक्स क


> **The single most common AI-assistant error on this platform.** Every coding assistant
> writes `client.chat.completions.create(...)` because that is what every other
> OpenAI-shaped SDK does. The native client breaks that convention.
> `npx skills add sarvamai/skills` fixes it permanently.

---
## 2 · `reasoning_effort` — the parameter that surprises everyone

In [5]:
import time
PROMPT = "A loan of ₹5,00,000 at 9.5% reducing balance over 60 months. Monthly EMI?"

for effort, budget in [(None, 400), ("low", 400), ("low", 3000)]:
    t0 = time.perf_counter()
    kw = dict(model="sarvam-105b", max_tokens=budget,
              messages=[{"role": "user", "content": PROMPT}])
    if effort is None:
        kw["reasoning_effort"] = None    # four possible effort levels: None, "low", "medium", "high"
    r = client.chat.completions(**kw)
    dt = time.perf_counter() - t0
    cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    content = r.choices[0].message.content
    print(f"effort={str(effort):<5} budget={budget:<5} {dt:>5.1f}s  "
          f"out={r.usage.completion_tokens:<5} "
          f"content={'None ⚠️' if content is None else content[:60]+'...'}")

effort=None  budget=400     6.3s  out=372   content=
I need to calculate the monthly EMI for a reducing balance ...
effort=low   budget=400     6.8s  out=400   content=None ⚠️
effort=low   budget=3000   53.1s  out=3000  content=None ⚠️


| Setting | Keeps reasoning | Cost | Speed | Use for |
|---|---|---|---|---|
| `reasoning_effort=None` | ❌ | lowest | fastest | classification, extraction, routing |
| `"low"` + generous `max_tokens` | ✅ | higher | slower | multi-step logic, agents, maths |

**Watch the billing.** Even when `content` is `None`, the reasoning tokens are billed.
A silent failure that costs money is worse than one that crashes.

---
## 3 · Every parameter, swept

In [6]:
BASE = dict(model="sarvam-105b", max_tokens=600, reasoning_effort=None,
            messages=[{"role": "user", "content": "Name three risks of an unsecured personal loan."}])

SWEEP = [
    ("temperature=0.0",   dict(temperature=0.0)),
    ("temperature=1.5",   dict(temperature=1.5)),
    ("top_p=0.3",         dict(top_p=0.3)),
    ("freq_penalty=1.5",  dict(frequency_penalty=1.5)),
    ("pres_penalty=1.5",  dict(presence_penalty=1.5)),
    ("stop=['3.']",       dict(stop=["3."])),
]

for label, kw in SWEEP:
    r = client.chat.completions(**{**BASE, **kw})
    cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    print(f"\n── {label}\n{(r.choices[0].message.content or '')[:220]}")


── temperature=0.0

Here are three risks of an unsecured personal loan:

1. **Higher interest rates** – Because the lender has no collateral to seize if you default, they charge higher rates than secured loans, increasing your total repaym

── temperature=1.5

Here are three key risks of an unsecured personal loan:

1. **Higher interest rates and fees** — Because the lender has no collateral to seize if you default, they compensate for the increased risk by charging higher in

── top_p=0.3

Here are three risks of an unsecured personal loan:

1. **Higher interest rates** – Because the loan isn't backed by collateral, lenders face greater risk and typically charge higher rates than secured loans, increasing

── freq_penalty=1.5

Here are three risks of an unsecured personal loan:

**1. Higher Interest Rates and Costs**
Since the lender bears more risk (no collateral to seize if you default), they charge higher interest rates compared to secured

── pres_penalty=1.5

Here are three key 

In [7]:
# seed = reproducibility. Essential for eval harnesses (Lab 07).
outs = []
for _ in range(2):
    r = client.chat.completions(**{**BASE, "temperature": 0.9, "seed": 42})
    cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    outs.append(r.choices[0].message.content)
print("identical with seed=42 :", outs[0] == outs[1])

identical with seed=42 : False


| Parameter | Range | Default | When you touch it |
|---|---|---|---|
| `temperature` | 0–2 | 0.5 (reasoning on) / 0.2 (off) | 0–0.3 extraction · 0.7+ generation |
| `top_p` | 0–1 | 1 | Leave alone unless you know why |
| `max_tokens` | — | — | Starter 4096 / Pro 16384 / Business 128000 |
| `frequency_penalty` | −2…2 | 0 | Long generation that repeats itself |
| `presence_penalty` | −2…2 | 0 | Force topic movement |
| `seed` | int | — | **Always set it in evals** |
| `stop` | ≤4 strings | — | Structured output boundaries |
| `n` | 1–128 | 1 | Sampling multiple candidates |
| `stream` | bool | False | Any user-facing interface |

---
## 4 · Streaming

In [8]:
t0 = time.perf_counter(); first = None; buf = []
for chunk in client.chat.completions(
        model="sarvam-105b", max_tokens=1500, stream=True,
        messages=[{"role": "user", "content": "भारत की डिजिटल क्रांति पर एक संक्षिप्त विश्लेषण।"}]):
    if chunk.choices:
        d = chunk.choices[0].delta
        if getattr(d, "content", None):
            if first is None:
                first = time.perf_counter() - t0
                print(f"[first token @ {first*1000:.0f} ms]\n")
            buf.append(d.content); print(d.content, end="", flush=True)
print(f"\n\ntotal {time.perf_counter()-t0:.1f}s · {len(''.join(buf))} chars")

[first token @ 21059 ms]


**भारत की डिजिटल क्रांति – एक नज़र**

|                     | मुख्य बातें |
|---------------------|------|
| **स्केल (Scale)**       | लगभग 75 करोड़ इंटरनेट यूज़र्स (2023); 60 करोड़ से ज़्यादा स्मार्टफोन सब्सक्राइबर; 1.14 अरब मोबाइल स्पॉन्सरशिप (टेलीकॉम)। |
| **ग्रोथ इंजन (Growth engine)** | 5 साल (2015-2020) में मोबाइल डेटा का खर्च 30 गुना गिर गया; 2021-23 की महामारी के दौरान फिक्स्ड-लाइन से मोबाइल पर जाना और तेज़ी से बढ़ गया। |
| **पॉलिसी बैकबोन (Policy backbone)** | डिजिटल इंडिया (2015), आधार इंटीग्रेटेड डिजिटल आइडेंटिटी (2009-वर्तमान), इंडिया स्टैक (Aadhaar, UPI, API-ड्रिवन सर्विसेज़), GST (2017) जिसने फॉर्मल-डिजिटलाइज़ेशन को बढ़ावा दिया। |
| **इम्पैक्ट पैन (Impact pane)** | फाइनेंस, रिटेल, एग्रीकल्चर, हेल्थ, एजुकेशन, गवर्नेंस और एंटरटेनमेंट। |
| **चैलेंजेज़ (Challenges)** | डिजिटल डिवाइड, साइबर सिक्योरिटी, डेटा-प्राइवेसी लेजिस्लेशन, लगातार बन रहे इंफ्रास्ट्रक्चर गैप। |
|

total 25.3s · 889 chars


### **HTTP Stream vs WebSocket — When to Use Which**

|  | HTTP Stream | WebSocket |
|---|---|---|
 | Protocol	 | Single POST request	 | Persistent bidirectional connection |
 | Setup	 | Zero — it’s a normal HTTP call	 | Handshake + config message before first text | 
 | Endpoint	 | /text-to-speech/stream	 | /text-to-speech/ws |  
 | Text input	 | One text payload per request	 | Send multiple texts on the same connection |  
 | Max text	 | 3500 characters	 | 2500 characters per message (send many) | 
  | Audio output	 | Binary stream (play/save directly)	 | Base64-encoded chunks (decode each one) |  
 | Connection reuse	 | New connection per request	 | One connection, many conversions |  
 | Best for	 | One-shot generation, server-side pipelines, simple integrations	 | Voice agents, interactive apps, multi-turn conversations |  

---
## 5 · Prompt caching — a 62% saving that is an architecture decision

Cached input is **₹10.98/1M** against **₹29.28/1M**. The trick is to put everything
stable *first* and keep it byte-identical between calls.

In [9]:
STABLE_SYSTEM = ("You are a loan servicing assistant for an Indian NBFC. "
                 "Answer only from the policy below. Be concise.\n\n"
                 + "POLICY:\n" + ("Late payment attracts 2% per month. "
                 "Prepayment allowed after 6 EMIs with 2% charge. " * 40))

def ask(user_msg):
    r = client.chat.completions(
        model="sarvam-105b", max_tokens=500, reasoning_effort=None,
        messages=[{"role": "system", "content": STABLE_SYSTEM},   # ← identical every time
                  {"role": "user",   "content": user_msg}])
    u = r.usage
    cached = getattr(u, "prompt_tokens_details", None)
    cached = getattr(cached, "cached_tokens", 0) if cached else 0
    cost.llm(u.prompt_tokens, u.completion_tokens, cached)
    return r.choices[0].message.content, u.prompt_tokens, cached

for i, q in enumerate(["What is the late payment charge?",
                       "Can I prepay after 3 EMIs?",
                       "What happens if I miss two EMIs?"]):
    ans, ptok, cached = ask(q)
    print(f"call {i+1}: prompt={ptok:>5} cached={cached:>5}  → {ans[:70]}")

call 1: prompt=  972 cached=    0  → 
Late payment attracts 2% per month.
call 2: prompt=  975 cached=    0  → 
No, prepayment is allowed only after 6 EMIs with a 2% charge.
call 3: prompt=  974 cached=    0  → 
Missing two EMIs will attract a late payment charge of 2% per month o


In [10]:
# What caching is worth at scale
SYS_TOKENS, TURNS, CALLS = 2000, 8, 100_000
uncached = SYS_TOKENS * TURNS * CALLS * 29.28 / 1_000_000
cached   = SYS_TOKENS * TURNS * CALLS * 10.98 / 1_000_000
print(f"uncached ₹{uncached:>12,.0f} / month")
print(f"cached   ₹{cached:>12,.0f} / month")
print(f"saving   ₹{uncached-cached:>12,.0f}  ({(1-cached/uncached):.0%})")

uncached ₹      46,848 / month
cached   ₹      17,568 / month
saving   ₹      29,280  (62%)


---
## 6 · Tool calling — the foundation of everything agentic

In [11]:
# ── Mock backend ──────────────────────────────────────────────────────────
ACCOUNTS = {"LN1001": {"name": "Rajesh Kumar", "emi": 12500, "due": "2026-08-15",
                       "outstanding": 340000, "overdue_days": 0}}

def get_account(account_id: str):
    return ACCOUNTS.get(account_id, {"error": "not found"})

def get_emi_schedule(account_id: str):
    a = ACCOUNTS.get(account_id)
    return {"error": "not found"} if not a else {
        "next_due": a["due"], "amount": a["emi"], "remaining_months": 28}

def raise_ticket(account_id: str, issue: str):
    return {"ticket_id": "TKT-88214", "status": "open", "issue": issue}

REGISTRY = {"get_account": get_account,
            "get_emi_schedule": get_emi_schedule,
            "raise_ticket": raise_ticket}

TOOLS = [
    {"type": "function", "function": {
        "name": "get_account", "description": "Fetch loan account details by account ID",
        "parameters": {"type": "object", "properties": {
            "account_id": {"type": "string", "description": "Loan account number, e.g. LN1001"}},
            "required": ["account_id"]}}},
    {"type": "function", "function": {
        "name": "get_emi_schedule", "description": "Fetch the upcoming EMI schedule",
        "parameters": {"type": "object", "properties": {
            "account_id": {"type": "string", "description": "Loan account number"}},
            "required": ["account_id"]}}},
    {"type": "function", "function": {
        "name": "raise_ticket", "description": "Raise a support ticket for an unresolved issue",
        "parameters": {"type": "object", "properties": {
            "account_id": {"type": "string", "description": "Loan account number"},
            "issue": {"type": "string", "description": "Short description of the problem"}},
            "required": ["account_id", "issue"]}}},
]
print("3 tools registered")

3 tools registered


In [12]:
def run_agent(user_msg, max_turns=5, verbose=True):
    msgs = [{"role": "system", "content":
             "You are a loan servicing agent. Use tools for any account fact. "
             "Never invent numbers. Reply in the user's language."},
            {"role": "user", "content": user_msg}]

    for turn in range(max_turns):
        r = client.chat.completions(model="sarvam-105b", messages=msgs,
                                    tools=TOOLS, max_tokens=2000)
        cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
        m = r.choices[0].message
        calls = getattr(m, "tool_calls", None)

        if not calls:
            return m.content

        msgs.append({"role": "assistant", "content": m.content, "tool_calls":
                     [{"id": c.id, "type": "function",
                       "function": {"name": c.function.name,
                                    "arguments": c.function.arguments}} for c in calls]})
        for c in calls:
            args = json.loads(c.function.arguments)
            out = REGISTRY[c.function.name](**args)
            if verbose:
                print(f"  🔧 {c.function.name}({args}) → {out}")
            msgs.append({"role": "tool", "tool_call_id": c.id,
                         "content": json.dumps(out, ensure_ascii=False)})
    return "max turns reached"


print(run_agent("मेरे अकाउंट LN1001 की अगली EMI कब है और कितनी है?"))

  🔧 get_account({'account_id': 'LN1001'}) → {'name': 'Rajesh Kumar', 'emi': 12500, 'due': '2026-08-15', 'outstanding': 340000, 'overdue_days': 0}
  🔧 get_emi_schedule({'account_id': 'LN1001'}) → {'next_due': '2026-08-15', 'amount': 12500, 'remaining_months': 28}
आपके अकाउंट LN1001 की अगली EMI **15 अगस्त 2026** को देय है।

**EMI की राशि:** ₹12,500

**कुल बकाया:** ₹3,40,000
**अवधि शेष:** 28 महीने


In [13]:
# Multi-tool: this one should call two tools then answer
print(run_agent("Account LN1001 — my auto-debit failed last month. "
                "What's my outstanding, and please raise a ticket."))

  🔧 get_account({'account_id': 'LN1001'}) → {'name': 'Rajesh Kumar', 'emi': 12500, 'due': '2026-08-15', 'outstanding': 340000, 'overdue_days': 0}
  🔧 get_emi_schedule({'account_id': 'LN1001'}) → {'next_due': '2026-08-15', 'amount': 12500, 'remaining_months': 28}
  🔧 raise_ticket({'account_id': 'LN1001', 'issue': 'Auto-debit failed last month'}) → {'ticket_id': 'TKT-88214', 'status': 'open', 'issue': 'Auto-debit failed last month'}
Your outstanding balance on **LN1001** is **₹3,40,000**.

I’ve raised a ticket for the auto-debit failure:
- **Ticket ID:** TKT-88214
- **Status:** Open

Your next EMI of **₹12,500** is due on **15 Aug 2026**. Let me know if you need further help.


---
## 7 · Structured output — JSON you can trust

In [14]:
SCHEMA_HINT = '''Return ONLY valid JSON matching:
{"intent": "<balance|emi|complaint|other>",
 "account_id": "<string or null>",
 "sentiment": "<positive|neutral|negative>",
 "urgency": <1-5>,
 "summary": "<one line in English>"}'''

def classify(msg):
    r = client.chat.completions(
        model="sarvam-105b", max_tokens=400, temperature=0, reasoning_effort=None,
        messages=[{"role": "system", "content": SCHEMA_HINT},
                  {"role": "user", "content": msg}])
    cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    raw = (r.choices[0].message.content or "").strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"_parse_error": raw[:200]}

for m in ["मेरा EMI तीन बार fail हो गया, बहुत परेशान हूँ! Account LN1001",
          "What is my current balance?"]:
    print(json.dumps(classify(m), ensure_ascii=False, indent=2))

{
  "intent": "emi",
  "account_id": "LN1001",
  "sentiment": "negative",
  "urgency": 4,
  "summary": "The user is frustrated because their EMI has failed three times and needs immediate assistance."
}
{
  "intent": "balance",
  "account_id": null,
  "sentiment": "neutral",
  "urgency": 2,
  "summary": "Customer is requesting their current account balance."
}


> **Production tip.** `temperature=0` + `reasoning_effort=None` + an explicit schema in
> the system prompt gives you parseable JSON far more reliably than asking nicely.
> Always wrap `json.loads` in a try — and log the raw string when it fails.

In [15]:
cost.report()

LLM          ₹   0.1174  21 in / 1595 out
LLM          ₹   0.1199  21 in / 1630 out
LLM          ₹   0.0285  42 in / 372 out
LLM          ₹   0.0304  39 in / 400 out
LLM          ₹   0.2207  39 in / 3000 out
LLM          ₹   0.0088  22 in / 111 out
LLM          ₹   0.0139  22 in / 181 out
LLM          ₹   0.0099  22 in / 127 out
LLM          ₹   0.0446  22 in / 600 out
LLM          ₹   0.0110  22 in / 142 out
LLM          ₹   0.0081  22 in / 102 out
LLM          ₹   0.0128  22 in / 166 out
LLM          ₹   0.0157  22 in / 206 out
LLM          ₹   0.0293  972 in / 11 out
LLM          ₹   0.0300  975 in / 20 out
LLM          ₹   0.0302  974 in / 23 out
LLM          ₹   0.0185  344 in / 115 out
LLM          ₹   0.0391  498 in / 335 out
LLM          ₹   0.0244  356 in / 191 out
LLM          ₹   0.0376  510 in / 310 out
LLM          ₹   0.1195  659 in / 1369 out
LLM          ₹   0.0081  99 in / 71 out
LLM          ₹   0.0055  87 in / 41 out
TOTAL        ₹   0.9840
              (₹1000 free 

0.98401296